# Cross Model Transferability Analysis

Analyze saved cross-model transferability CSVs. This notebook uses only stored outputs and keeps query alignment assumptions explicit.

## Summary Statistics

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style='whitegrid')
ROOT = Path('..').resolve()
RUN_ROOT = ROOT / 'outputs' / 'cross_model_eval'

def latest_run_dir(base: Path) -> Path:
    runs = sorted(p for p in base.glob('run_*') if p.is_dir())
    if not runs:
        raise FileNotFoundError(f'No runs found under {base}')
    return runs[-1]

run_dir = latest_run_dir(RUN_ROOT)
raw_path = run_dir / 'raw_results.csv'
detailed_path = run_dir / 'detailed_results.csv'
df = pd.read_csv(raw_path)
df_detailed = pd.read_csv(detailed_path)

print('Using run:', run_dir)
print('Total rows:', len(df))
print(df.groupby('source_model').size())
print('Detailed rows:', len(df_detailed))

## Visualization

Transferability matrices summarize mean and variability across suffixes for each source/target pair.

In [ ]:
pivot = df.groupby(['source_model', 'target_model'])['success_rate'].mean().unstack()
std_matrix = df.groupby(['source_model', 'target_model'])['success_rate'].std().unstack()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis', ax=axes[0], cbar_kws={'label': 'Mean success rate'})
axes[0].set_title('Mean Transferability')
axes[0].set_xlabel('Target Model')
axes[0].set_ylabel('Source Model')

sns.heatmap(std_matrix, annot=True, fmt='.3f', cmap='magma', ax=axes[1], cbar_kws={'label': 'Std. dev. success rate'})
axes[1].set_title('Transferability Variability')
axes[1].set_xlabel('Target Model')
axes[1].set_ylabel('Source Model')

plt.tight_layout()
plt.show()

In [ ]:
for model in df_detailed['source_model'].dropna().unique():
    sub = df_detailed[df_detailed['source_model'] == model]
    pivot = sub.pivot_table(index='query_id', columns='target_model', values='success')
    plt.figure(figsize=(8, max(4, 0.25 * len(pivot))))
    sns.heatmap(pivot, cmap='viridis', annot=True, fmt='.0f', cbar_kws={'label': 'Success'})
    plt.title(f'Per-Query Transferability: {model}')
    plt.xlabel('Target Model')
    plt.ylabel('Query ID')
    plt.tight_layout()
    plt.show()

query_success = df_detailed.groupby('query_id')['success'].sum()
plt.figure(figsize=(8, 4))
sns.histplot(query_success, bins=10, kde=False, color='steelblue')
plt.title('Query-Level Robustness Across Targets')
plt.xlabel('Number of Successful Target Models')
plt.ylabel('Count of Queries')
plt.tight_layout()
plt.show()

## Insights

- Which source model transfers best can be inspected from the mean heatmap.
- Which queries are universally vulnerable appear as higher-count bins in the robustness histogram and full rows in the per-query heatmaps.
- Results are based on available queries; subset consistency assumed.